<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Appendix E: Batching and throughput-oriented execution

Packages that are being used in this notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- Throughout the main chapters, we usually process one example at a time
- This keeps the code compact and easier to understand
- But also, the code is already very expensive to run, so adding batching support would add little benefit due to hardware and resource limitations
- However, in certain contexts, having the ability to run the code in batched mode is still useful
- This appendix explains the broad idea behind batched execution and shows how to use it for the different chapters using code from the supplementary materials

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 Why batching helps

- There are two different performance goals:
  - latency: how quickly we get the answer for a single prompt;
  - throughput: how many prompts we can process in a given amount of time.
- Single-example generation is often best for minimizing latency and for debugging code
- Batching targets throughput primarily
- If we want to evaluate hundreds of problems on MATH-500, generate many self-consistency samples, or train on many supervised examples, batching can reduce the total runtime substantially on suitable hardware
  - That said, batching is not guaranteed to be faster on every device
  - Small models on CPUs or some less optimized GPUs may not benefit from batching; we may even get slowdowns, because the additional padding and batching overhead can offset the gains from parallelism

&nbsp;
## E.2 Running batched generation

- The main technical obstacle in batching is that prompts usually have different lengths
- For example, one math problem may tokenize to 40 tokens while another may tokenize to 120 tokens
- Since tensors in PyTorch must have rectangular shapes, we pad the shorter sequences so they all fit into a single batch tensor

- Conceptually, this makes batched generation much more difficult to implement than single-prompt generation
- In the main chapter, we used the `Qwen3Model` class from `reasoning_from_scratch.qwen3` (which uses the Qwen3 implementation explained in appendix C)
- For batched generation, since we have to keep track of padding tokens, etc., there is a separate `Qwen3Model` class in `reasoning_from_scratch.qwen3_batched` (the source code can be viewed in the supplementary materials at https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py)

- To illustrate the usage of the batched generation utilities, let's take a look at a concrete example
- We start with a single-sequence text generation example similar to what we have used in the main chapters
- Here, were apply it to two prompts (`["2+2?", "3+3=6?"]`) sequentially:

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- Below, we will use similar code from `reasoning_from_scratch.qwen3_batched` that supports batching
- Note that the batched version does not support streaming, though, meaning we have to wait until all results are generated before they are decoded and printed
- Here, the batched generation uses left padding, which will be explained in the next section
- For now, let's start with a usage example to illustrate how it's used (before we get into how it works internally)

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- As we can see, the results are exactly the same as before
- The difference is that these results were generated in parallel via `generate_text_basic_batched_cache`
- The next section briefly explains how this works under the hood

- An even more optimized code implementation replaces `generate_text_basic_batched_cache` with `generate_text_basic_batched_cache_stop`
- `generate_text_basic_batched_cache` keeps every row in the active batch for every decode step 
- `generate_text_basic_batched_cache_stop`removes finished rows from the active compute batch (it's more complicated to implement internally, but can optimize performance
- This is illustrated in the figure below

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- Side note: in Qwen3, the The `<eos>` tokens are `<|endoftext|>`, but the figure uses `<eos>` for visual compactness

&nbsp;
## E.3 Padding and attention masks

- In single-example mode, if we tokenize a short prompt such as `"2+2?"`, we can pass it to the model as a simple tensor of shape `(1, 4)`:
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- Internally, the model builds a standard causal attention mask internally so that each position can only attend to itself and earlier tokens
- If you are unfamiliar with self-attention, I have an article that provides more background information: https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- Conceptually, that mask looks like this:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` means "masked out" and `0` means "allowed"
- So the first token cannot look ahead to later positions, the second token can only look at the first two positions, and so on
- This is the standard autoregressive masking pattern

- Batching changes the situation because different prompts usually have different lengths
- Suppose we process `"2+2?"` together with the slightly longer prompt `"3+3=6?"`
- Since PyTorch tensors must be rectangular, the shorter row has to be padded to match the longer one
- Here, this is done with left padding:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- Note that we keep an additional `attn_mask` internally; this is just to keep track off the padded positions
- In this `attn_mask`, `True` means padded and `False` means not padded
- We use this additional `attn_mask` to identify the tokens in the causal mask that correspond to the pad token IDs
- Masking padded keys and zeroing padded queries are important steps to make batching behave similarly to the single-example execution

- By the way, we use the `<|endoftext|>` token, but it does not really matter because the corresponding token positions are ignored anyways

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 Chapter 3: batched MATH-500 evaluation

- The supplementary materials includes a script for the evaluation method implemented in chapter 3 that we can download and use similar to how we did it in chapter 6:

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- Then, to run it, we can execute the following command in a code terminal (replace `uv run` with `python` if you are not a uv user):

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- The bonus material also includes a version of this for batched generation that applies the batching method we discussed previously
- The download is similar to before, except that we replace `evaluate_math500.py` with `evaluate_math500_batched.py`

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- The usage is also similar to the non-batched version, except that we now provide an additional `--batch_size` argument to specify how many prompts and answers the LLM should process in parallel

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- The ideal batch size depends on what your hardware can handle; a batch size of 64 uses approximately 23.39 GB RAM (the non-batched script uses approximately 1.84 GB RAM)
- We will compare and discuss the performance difference towards the end of the appendix

&nbsp;
## E.5 Chapter 4: batched self-consistency sampling

- The optional `self_consistency_math500_batched.py` script that implements self-consistency sampling in chapter 4 does not mix different prompts into one padded tensor
-  Instead, it repeats the same prompt `num_samples` times and samples several continuations in parallel for self-consistency voting
- Because every row starts from the same prompt length, this script uses the regular `Qwen3Model` from reasoning_from_scratch.qwen3 instead of reasoning_from_scratch.qwen3_batched, since padding is not needed for equal prompt lengths

- We can download the script as follows:

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- To download the non-batched version, simply drop the `"_batched"` in the file name above
- We can run the script as follows (the syntax for the non-batched script is identical)

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- More about the performance at the end of this appendix

&nbsp;
## E.6 Chapter 6: batched GRPO rollouts

- Self-refinement in chapter 5 is a sequential technique that itself does not benefit from batching
- One could run self-refinement loops for multiple inputs in parallel, but this is non-trivial to implement and thus not part of the supplementary material
- Instead, we continue with a batched version of RLVR in chapter 6
- In chapter 6, we use the same prompt for the different rollouts; hence, no padding is required here; so, similar to section E.5, the code uses the regular `Qwen3Model` class from `reasoning_from_scratch.qwen3`
- The relevant scripts can be fetched with:

In [ ]:
# Non-batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# Batched version
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# Batched version with GPU support
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- In the current script, `--batch_size` controls how many rollouts are generated in parallel within a step
- This increases throughput, but it also increases memory pressure, so in practice you may need to reduce `--num_rollouts` or `--max_new_tokens`
- If you have multiple GPUs, the FSDP variant follows the same pattern and adds `--num_gpus`
- Again, we will return to the performance discussion at the end of this appendix
- As of this writing, batched versions of the chapter 7 scripts are not available in the supplementary materials yet, but will be added over time; conceptually, they will work similarly to the chapter 6 scripts

&nbsp;
## E.7 Chapter 8: batched distillation

- Chapter 8 returns to the padding-aware style from chapter 3, since distillation examples have different prompt and answer lengths
- You can download the script and sample training dataset as follows:

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- For the non-batched version, drop the `"_batched"` in the file name
- We can run the script as follows:

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 Single-sequence versus batch generation

- The table below summarizes the runtime and RAM usage numbers for the scripts above

| Row | Script                                   | Batch size | RAM      | H100 Total time (min) | DGX Spark Total time (min) |
|-----|------------------------------------------|------------|----------|------------------------|-----------------------------|
| 1   | evaluate_math500.py                      | -          | 1.8 GB   | 90.0                   | 174.7                       |
| 2   | evaluate_math500_batched.py              | 64         | 23.39 GB | 16.0                   | 108.4                       |
|     |                                          |            |          |                        |                             |
| 3   | self_consistency_math500.py              | -          | 1.79 GB  | 252.0                  | 340.8                       |
| 4   | self_consistency_math500_batched.py      | 3          | 2.45 GB  | 129.0                  | 243.3                       |
|     |                                          |            |          |                        |                             |
| 5   | rlvr_grpo_original_no_kl.py              | -          | 43.35 GB | 68.0                   | 63.7                        |
| 6   | rlvr_grpo_original_no_kl_batched.py      | 4          | 44.91 GB | 19.0                   | 23.1                        |
|     |                                          |            |          |                        |                             |
| 7   | distill.py                              | -          | 8.29 GB  | 10.9                   | 32.8                        |
| 8   | distill_batched.py                      | 4          | 8.34 GB  | 9.1                    | 28.2                        |